In [0]:
dbutils.widgets.text("catalog_name", "allianz_coe", "Catalog Name")
dbutils.widgets.text("gold_schema_name", "gold_c360", "Gold Schema Name")
dbutils.widgets.text("vault_schema_name", "dvault_faker01", "Vault Schema Name")

catalog_name = dbutils.widgets.get("catalog_name")
gold_schema_name = dbutils.widgets.get("gold_schema_name")
vault_schema_name = dbutils.widgets.get("vault_schema_name")

In [0]:
DIM_PRODUCT_CFG = {
  "name": "dim_product",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_product",
  "business_key_col": "product_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: product_id, effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
      h.product_id AS product_id,
      s.type       AS product_type,
      to_timestamp(s.load_date) AS effective_from_ts,
      CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
    FROM {catalog_name}.{vault_schema_name}.hub_product h
    JOIN {catalog_name}.{vault_schema_name}.sat_product s
      ON s.product_hash_key = h.product_hash_key
    WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

  "attribute_cols": ["product_id", "product_type"],

  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "product_id",
    "product_type",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_IDENTITY_CFG = {
  "name": "dim_identity",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_identity",
  "business_key_col": "identity_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
      h.identities_id       AS identity_id,
      s.ecid       AS experience_cloud_id,
      s.hashed_email       AS email_address_hash_value,
      to_timestamp(s.load_date) AS effective_from_ts,
      CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
    FROM {catalog_name}.{vault_schema_name}.hub_identities h
    JOIN {catalog_name}.{vault_schema_name}.sat_identities s
      ON s.identities_hash_key = h.identities_hash_key
    WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY identity_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

  "attribute_cols": ["identity_id","experience_cloud_id","email_address_hash_value"],
 
  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "identity_id",
    "experience_cloud_id",
    "email_address_hash_value",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_CUSTOMER_CFG = {
  "name": "dim_customer",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_customer",
  "business_key_col": "customer_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
      h.customer_id       AS customer_id,
      s.customer_number       AS customer_number,
      s.customer_status       AS customer_status_code,
      s.customer_status_reason AS customer_status_reason,
      s.customer_since AS customer_since_date,
      s.customer_rating AS customer_rating,
      s.customer_segment AS customer_segment,
      s.line_of_business AS line_of_business,
      s.nps_score AS net_promoter_score,
      to_timestamp(s.load_date) AS effective_from_ts,
      CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
    FROM {catalog_name}.{vault_schema_name}.hub_customer h
    JOIN {catalog_name}.{vault_schema_name}.sat_customer s
      ON s.customer_hash_key = h.customer_hash_key
    WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

  "attribute_cols": ["customer_id","customer_number","customer_status_code","customer_status_reason","customer_since_date","customer_rating","customer_segment","line_of_business","net_promoter_score"],
 
  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "customer_id",
    "customer_number",
    "customer_status_code",
    "customer_status_reason",
    "customer_since_date",
    "customer_rating",
    "customer_segment",
    "line_of_business",
    "net_promoter_score",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_ACCOUNT_CFG = {
  "name": "dim_account",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_account",
  "business_key_col": "account_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
      h.account_id       AS account_id,
      s.account_number       AS account_number,
      s.account_type       AS account_type,
      s.account_last_access AS account_last_access_ts,
      s.account_last_change AS account_last_change_ts,
      s.account_creation_type AS account_creation_type,
      s.account_status AS account_status,
      to_timestamp(s.load_date) AS effective_from_ts,
      CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
    FROM {catalog_name}.{vault_schema_name}.hub_account h
    JOIN {catalog_name}.{vault_schema_name}.sat_account s
      ON s.account_hash_key = h.account_hash_key
    WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

  "attribute_cols": ["account_id","account_number","account_type","account_last_access_ts","account_last_change_ts","account_creation_type","account_status"],
 
  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "account_id",
    "account_number",
    "account_type",
    "account_last_access_ts",
    "account_last_change_ts",
    "account_creation_type",
    "account_status",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_POLICY_CFG = {
  "name": "dim_policy",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_policy",
  "business_key_col": "policy_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
  h.policy_id                              AS policy_id,
  s.cover_option                           AS policy_cover_option,
  s.declined_claims                        AS declined_claims_number,
  s.fraud_flag                             AS is_fraud,
  --s.gross_revenue                          AS gross_revenue,
  --s.net_revenue                            AS net_revenue,
  s.number_of_active_claim                 AS active_claims_number,
  s.number_of_previous_claim               AS previous_claims_number,
  s.policy_cycle                           AS policy_cycle,
  s.policy_end_date                        AS policy_end_ts,
  s.policy_length                          AS policy_tennure,
  s.policy_number                          AS policy_number,
  s.policy_start_date                      AS policy_start_ts,
  s.policy_status                          AS policy_status,
  hpr.product_id                           AS product_id,
  spr.type                                 AS product_type,
  --s.renewal_amount_current_period          AS renewal_amount_current_period,
  --s.renewal_amount_next_period             AS renewal_amount_next_period,
  s.renewal_date                           AS renewal_date,
  s.sales_channel                          AS policy_sales_channel,
  to_timestamp(s.load_date)                AS effective_from_ts,
  CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
FROM {catalog_name}.{vault_schema_name}.hub_policy h
JOIN {catalog_name}.{vault_schema_name}.sat_policy s
  ON s.policy_hash_key = h.policy_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.link_policy_product rpp
  ON rpp.policy_hash_key = h.policy_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_product hpr
  ON hpr.product_hash_key = rpp.product_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.sat_product spr
  ON spr.product_hash_key = hpr.product_hash_key
WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY policy_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

"attribute_cols": [
    "policy_id",
    "policy_cover_option",
    "declined_claims_number",
    "is_fraud",
    # "gross_revenue",
    # "net_revenue",
    "active_claims_number",
    "previous_claims_number",
    "policy_cycle",
    "policy_end_ts",
    "policy_tennure",
    "policy_number",
    "policy_start_ts",
    "policy_status",
	  "product_id",
	  "product_type",
    # "renewal_amount_current_period",
    # "renewal_amount_next_period",
    "renewal_date",
    "policy_sales_channel"
],
 
  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "policy_id",
    "policy_cover_option",
    "declined_claims_number",
    "is_fraud",
    "active_claims_number",
    "previous_claims_number",
    "policy_cycle",
    "policy_end_ts",
    "policy_tennure",
    "policy_number",
    "policy_start_ts",
    "policy_status",
	  "product_id",
	  "product_type",
    "renewal_date",
    "policy_sales_channel",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_MOTOR_CFG = {
  "name": "dim_motor",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_motor",
  "business_key_col": "motor_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: motor_id, effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
        s2.product_sk AS product_sk,
        h.motor_id AS motor_id,
        s.auto_decline_vehicle        AS is_auto_decline_vehicle,
        s.body_type                   AS vehicle_body_type,
        s.fuel_type                   AS vehicle_fuel_type,
        s.license_status              AS driver_license_status,
        s.is_existing_motor_customer  AS is_existing_motor_customer,
        s.motor_lapsed_policies       AS motor_lapsed_policies,
        s.motor_risk_address          AS vehicle_risk_address,
        s.risk_class_code             AS vehicle_risk_class_code,
        s.variant                     AS vehicle_variant,
        s.vehicle_owner_type          AS vehicle_owner_type,
        s.vehicle_regstate            AS vehicle_reg_state,
        s.vehicle_class               AS vehicle_class,
        s.vehicle_model               AS vehicle_model,
        s.vehicle_type                AS vehicle_type,
        s.motor_sum_insrd             AS vehicle_sum_insured_amt,
        s.vehicle_year                AS vehicle_year,
        s.vehicle_age                 AS vehicle_age,
        to_timestamp(s.load_date) AS effective_from_ts,
        CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
    FROM {catalog_name}.{vault_schema_name}.hub_motor h
    JOIN {catalog_name}.{vault_schema_name}.sat_motor s
      ON s.motor_hash_key = h.motor_hash_key
    JOIN {catalog_name}.{gold_schema_name}.dim_product s2
      ON s2.product_type = 'PRD_MOTOR_PERSONAL' and s2.effective_to_ts = CAST('9999-12-31 00:00:00' AS TIMESTAMP)
    WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY motor_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

  "attribute_cols": [
    "product_sk",
    "motor_id",
    "is_auto_decline_vehicle",
    "vehicle_body_type",
    "vehicle_fuel_type",
    "driver_license_status",
    "is_existing_motor_customer",
    "motor_lapsed_policies",
    "vehicle_risk_address",
    "vehicle_risk_class_code",
    "vehicle_variant",
    "vehicle_owner_type",
    "vehicle_reg_state",
    "vehicle_class",
    "vehicle_model",
    "vehicle_type",
    "vehicle_sum_insured_amt",
    "vehicle_year",
    "vehicle_age"
    ],

  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "product_sk",
    "motor_id",
    "is_auto_decline_vehicle",
    "vehicle_body_type",
    "vehicle_fuel_type",
    "driver_license_status",
    "is_existing_motor_customer",
    "motor_lapsed_policies",
    "vehicle_risk_address",
    "vehicle_risk_class_code",
    "vehicle_variant",
    "vehicle_owner_type",
    "vehicle_reg_state",
    "vehicle_class",
    "vehicle_model",
    "vehicle_type",
    "vehicle_sum_insured_amt",
    "vehicle_year",
    "vehicle_age",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_HOME_CFG = {
  "name": "dim_home",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_home",
  "business_key_col": "home_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  # Stage SQL must output: home_id, effective_from_ts, attributes...
  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    base AS (
    SELECT 
        s2.product_sk AS product_sk,
        h.home_id                       AS home_id,
        s.wall_construction             AS wall_construction_material_type,
        s.home_risk_address             AS home_risk_address,
        s.roof_construction             AS roof_construction_material_type,
        s.home_type                     AS home_type,
        s.home_state                    AS home_state,
        s.is_existing_home_customer     AS is_existing_home_customer,
        to_timestamp(s.load_date)       AS effective_from_ts,
        CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts
    FROM {catalog_name}.{vault_schema_name}.hub_home h
    JOIN {catalog_name}.{vault_schema_name}.sat_home s
      ON s.home_hash_key = h.home_hash_key
    JOIN {catalog_name}.{gold_schema_name}.dim_product s2
      ON s2.product_type = 'PRD_HOME_PERSONAL' and s2.effective_to_ts = CAST('9999-12-31 00:00:00' AS TIMESTAMP)
    WHERE to_timestamp(s.load_date) > (SELECT wm FROM wm)),
    ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY home_id
            ORDER BY effective_from_ts DESC
        ) AS rnk
    FROM base
)

SELECT * EXCEPT(rnk)
FROM ranked
WHERE rnk = 1
  """,

  "attribute_cols": [
    "product_sk",
    "home_id",
    "wall_construction_material_type",
    "home_risk_address",
    "roof_construction_material_type",
    "home_type",
    "home_state",
    "is_existing_home_customer"
    ],

  # Columns to insert into target (explicit list keeps it stable)
  "insert_cols": [
    "product_sk",
    "home_id",
    "wall_construction_material_type",
    "home_risk_address",
    "roof_construction_material_type",
    "home_type",
    "home_state",
    "is_existing_home_customer",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}

In [0]:
DIM_PERSON_CFG = {
  "name": "dim_person",
  "target_table": f"{catalog_name}.{gold_schema_name}.dim_person",
  "business_key_col": "person_id",
  "watermark_col": "effective_from_ts",
  "effective_from_col": "effective_from_ts",
  "record_version_col": "record_version",

  "stage_sql": f"""
    WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),
    latest_person AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT sp.*,
               ROW_NUMBER() OVER (PARTITION BY person_hash_key ORDER BY load_date DESC) rn
        FROM {catalog_name}.{vault_schema_name}.sat_person sp
      ) WHERE rn = 1
    ),
    latest_nat AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT sn.*,
               ROW_NUMBER() OVER (PARTITION BY natural_person_hash_key ORDER BY load_date DESC) rn
        FROM {catalog_name}.{vault_schema_name}.sat_natural_person sn
      ) WHERE rn = 1
    ),
    latest_leg AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT sl.*,
               ROW_NUMBER() OVER (PARTITION BY legal_person_hash_key ORDER BY load_date DESC) rn
        FROM {catalog_name}.{vault_schema_name}.sat_legal_person sl
      ) WHERE rn = 1
    ),
    latest_contact AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT sc.*,
               ROW_NUMBER() OVER (PARTITION BY contact_hash_key ORDER BY load_date DESC) rn
        FROM {catalog_name}.{vault_schema_name}.sat_contact sc
      ) WHERE rn = 1
    ),
    latest_home_addr AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT sha.*,
               ROW_NUMBER() OVER (PARTITION BY home_address_hash_key ORDER BY load_date DESC) rn
        FROM {catalog_name}.{vault_schema_name}.sat_home_address sha
      ) WHERE rn = 1
    ),
    latest_consent AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT scon.*,
               ROW_NUMBER() OVER (PARTITION BY consent_hash_key ORDER BY load_date DESC) rn
        FROM {catalog_name}.{vault_schema_name}.sat_consent scon
      ) WHERE rn = 1
    ),

    -- Enriched person view from DV
    person_enriched AS (
      SELECT
        COALESCE(hp.person_id, hnp.natural_person_id, hlp.legal_person_id) AS person_id,

        -- Natural person fields
        sn.courtesy_title,
        sn.first_name,
        sn.last_name,
        sn.full_name,
        sn.birth_date,
        sn.gender,
        sn.nationality,
        sn.marital_status,
        sn.occupation,
        sn.job_title,
        sn.role,
        sn.preferred_language,
        sn.assesed_disability_degree as assessed_disability_degree,

        -- Person fields
        sp.type AS person_type,
        sp.is_lead,
        sp.source_id,
        sp.source_type,
        sp.tenant_id,
        sp.operational_paperless_consent AS is_operational_paperless_consent,

        -- Legal person fields
        sl.company_name,
        CAST(NULL AS DATE) AS date_of_constitution, -- map if you have a column

        -- Address hub+sat
        hha.home_address_id,
        sha.street AS street_address,
        sha.postcode,
        sha.city,
        sha.state,
        sha.country,

        -- Contact hub+sat
        hc.contact_id,
        sc.home_phone AS home_phone_number,
        sc.work_phone AS work_phone_number,
        sc.personal_email,
        sc.work_email,

        -- Consent hub+sat
        hcs.consent_id,
        scon.opt_in_legitimate_interest AS is_opt_in_legitimate_interest,
        scon.opt_in_validated           AS is_opt_in_validated,

        -- compute effective date driver as greatest DV load_date across contributing sats
        greatest(
          COALESCE(sp.load_date,  DATE('1900-01-01')),
          COALESCE(sn.load_date,  DATE('1900-01-01')),
          COALESCE(sl.load_date,  DATE('1900-01-01')),
          COALESCE(sc.load_date,  DATE('1900-01-01')),
          COALESCE(sha.load_date, DATE('1900-01-01')),
          COALESCE(scon.load_date,DATE('1900-01-01'))
        ) AS eff_date

      FROM {catalog_name}.{vault_schema_name}.hub_person hp
      LEFT JOIN latest_person sp
        ON sp.person_hash_key = hp.person_hash_key

      -- Natural subtype
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_natural_person lpn
        ON lpn.person_hash_key = hp.person_hash_key
      LEFT JOIN {catalog_name}.{vault_schema_name}.hub_natural_person hnp
        ON hnp.natural_person_hash_key = lpn.natural_person_hash_key
      LEFT JOIN latest_nat sn
        ON sn.natural_person_hash_key = lpn.natural_person_hash_key

      -- Legal subtype
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_legal_person lpl
        ON lpl.person_hash_key = hp.person_hash_key
      LEFT JOIN {catalog_name}.{vault_schema_name}.hub_legal_person hlp
        ON hlp.legal_person_hash_key = lpl.legal_person_hash_key
      LEFT JOIN latest_leg sl
        ON sl.legal_person_hash_key = lpl.legal_person_hash_key

      -- Contact
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_contact lpc
        ON lpc.person_hash_key = hp.person_hash_key
      LEFT JOIN {catalog_name}.{vault_schema_name}.hub_contact hc
        ON hc.contact_hash_key = lpc.contact_hash_key
      LEFT JOIN latest_contact sc
        ON sc.contact_hash_key = lpc.contact_hash_key

      -- Home address
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_home_address lpha
        ON lpha.person_hash_key = hp.person_hash_key
      LEFT JOIN {catalog_name}.{vault_schema_name}.hub_home_address hha
        ON hha.home_address_hash_key = lpha.home_address_hash_key
      LEFT JOIN latest_home_addr sha
        ON sha.home_address_hash_key = lpha.home_address_hash_key

      -- Consent
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_consent lps
        ON lps.person_hash_key = hp.person_hash_key
      LEFT JOIN {catalog_name}.{vault_schema_name}.hub_consent hcs
        ON hcs.consent_hash_key = lps.consent_hash_key
      LEFT JOIN latest_consent scon
        ON scon.consent_hash_key = lps.consent_hash_key
    )

    -- Final stage with geography_sk + identity_sk lookups
    SELECT DISTINCT
      -- geography_sk from dim_geography by city+state
      COALESCE(dg.geography_sk, -1) AS geography_sk,

      -- identity_sk from dim_identity by md5(personal_email)
      COALESCE(di.identity_sk, -1) AS identity_sk,

      pe.person_id,

      pe.courtesy_title,
      pe.first_name,
      pe.last_name,
      pe.full_name,
      pe.person_type,
      pe.birth_date,
      pe.gender,
      pe.nationality,
      pe.marital_status,
      pe.occupation,

      pe.home_address_id,
      pe.street_address,
      pe.postcode,

      pe.contact_id,
      pe.home_phone_number,
      pe.work_phone_number,
      pe.personal_email,
      pe.work_email,

      pe.job_title,
      pe.role,

      pe.company_name,
      pe.date_of_constitution,

      pe.is_lead,
      pe.preferred_language,
      pe.source_id,
      pe.source_type,
      pe.tenant_id,
      pe.assessed_disability_degree as assessed_disability_degree,
      pe.is_operational_paperless_consent,

      pe.consent_id,
      pe.is_opt_in_legitimate_interest,
      pe.is_opt_in_validated,

      to_timestamp(pe.eff_date) AS effective_from_ts,
      CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts

    FROM person_enriched pe

    -- dim_geography lookup (city + state)
    LEFT JOIN {catalog_name}.{gold_schema_name}.dim_geography dg
      ON upper(trim(dg.city))  = upper(trim(pe.city))
     AND upper(trim(dg.state)) = upper(trim(pe.state))
     AND upper(trim(dg.country)) = upper(trim(pe.country))
     --AND dg.is_current = true

    -- dim_identity lookup (md5(personal_email))
    LEFT JOIN {catalog_name}.{gold_schema_name}.dim_identity di
      ON di.email_address_hash_value = md5(lower(trim(pe.personal_email)))
     AND di.effective_to_ts = CAST('9999-12-31 00:00:00' AS TIMESTAMP)

    WHERE pe.eff_date > (SELECT wm FROM wm)
      AND pe.person_id IS NOT NULL
  """,

  # Hashdiff inputs (exclude effective dates; include SKs if you want them to trigger SCD2 changes)
  "attribute_cols": [
    "geography_sk",
    "identity_sk",
    "person_id",
    "courtesy_title",
    "first_name",
    "last_name",
    "full_name",
    "person_type",
    "birth_date",
    "gender",
    "nationality",
    "marital_status",
    "occupation",
    "home_address_id",
    "street_address",
    "postcode",
    "contact_id",
    "home_phone_number",
    "work_phone_number",
    "personal_email",
    "work_email",
    "job_title",
    "role",
    "company_name",
    "date_of_constitution",
    "is_lead",
    "preferred_language",
    "source_id",
    "source_type",
    "tenant_id",
    "assessed_disability_degree",
    "is_operational_paperless_consent",
    "consent_id",
    "is_opt_in_legitimate_interest",
    "is_opt_in_validated"
  ],

  # Insert list (includes record_version because your loader now computes it)
  "insert_cols": [
    "geography_sk",
    "identity_sk",
    "person_id",
    "courtesy_title",
    "first_name",
    "last_name",
    "full_name",
    "person_type",
    "birth_date",
    "gender",
    "nationality",
    "marital_status",
    "occupation",
    "home_address_id",
    "street_address",
    "postcode",
    "contact_id",
    "home_phone_number",
    "work_phone_number",
    "personal_email",
    "work_email",
    "job_title",
    "role",
    "company_name",
    "date_of_constitution",
    "is_lead",
    "preferred_language",
    "source_id",
    "source_type",
    "tenant_id",
    "assessed_disability_degree",
    "is_operational_paperless_consent",
    "consent_id",
    "is_opt_in_legitimate_interest",
    "is_opt_in_validated",
    "attr_hash",
    "effective_from_ts",
    "effective_to_ts",
    #"is_current",
    "record_version",
    "created_by",
    "created_ts",
    "last_updated_by",
    "last_updated_ts"
  ]
}


In [0]:
DIM_MARKETING_CFG = {
   "name": "dim_marketing",
   "target_table": f"{catalog_name}.{gold_schema_name}.dim_marketing",
   "business_key_cols": ["marketing_preference_id", "marketing_engagement_id"],
   "watermark_col": "effective_from_ts",
   "effective_from_col": "effective_from_ts",
   "record_version_col": "record_version",

   "stage_sql": f"""
  WITH wm AS (SELECT TIMESTAMP('${{WATERMARK}}') AS wm),

  -- -----------------------------
  -- Latest Lead per Person
  -- -----------------------------
  lead_latest AS (
    SELECT * EXCEPT(rn) FROM (
      SELECT
        lpl.person_hash_key,
        sl.preferred_contact_method,
        sl.load_date AS lead_load_date,
        ROW_NUMBER() OVER (
          PARTITION BY lpl.person_hash_key
          ORDER BY sl.load_date DESC
        ) AS rn
      FROM {catalog_name}.{vault_schema_name}.link_person_lead lpl
      JOIN {catalog_name}.{vault_schema_name}.sat_lead sl
        ON sl.lead_hash_key = lpl.lead_hash_key
    ) WHERE rn = 1
  ),

  -- -----------------------------
  -- Latest Marketing Preference link per Person
  -- -----------------------------
  pref_link_latest AS (
    SELECT * EXCEPT(rn) FROM (
      SELECT
        hp.person_hash_key,
        lpmp.marketing_preference_hash_key,
        ROW_NUMBER() OVER (
          PARTITION BY hp.person_hash_key
          ORDER BY lpmp.load_date DESC
        ) AS rn
      FROM {catalog_name}.{vault_schema_name}.hub_person hp
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
        ON lpmp.person_hash_key = hp.person_hash_key
    ) WHERE rn = 1
  ),

  -- Latest SAT Marketing Preference per preference hash key
  pref_sat_latest AS (
    SELECT * EXCEPT(rn) FROM (
      SELECT
        smp.*,
        ROW_NUMBER() OVER (
          PARTITION BY smp.marketing_preference_hash_key
          ORDER BY smp.load_date DESC
        ) AS rn
      FROM {catalog_name}.{vault_schema_name}.sat_marketing_preference smp
    ) WHERE rn = 1
  ),

  -- Preference hub for preference_id
  pref_hub AS (
    SELECT
      hmp.marketing_preference_hash_key,
      hmp.marketing_preference_id
    FROM {catalog_name}.{vault_schema_name}.hub_marketing_preference hmp
  ),

  -- -----------------------------
  -- Latest Marketing Engagement link per Person
  -- -----------------------------
  eng_link_latest AS (
    SELECT * EXCEPT(rn) FROM (
      SELECT
        hp.person_hash_key,
        lpme.marketing_engagement_hash_key,
        ROW_NUMBER() OVER (
          PARTITION BY hp.person_hash_key,lpme.marketing_engagement_hash_key
          ORDER BY lpme.load_date DESC
        ) AS rn
      FROM {catalog_name}.{vault_schema_name}.hub_person hp
      LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
        ON lpme.person_hash_key = hp.person_hash_key
    ) WHERE rn = 1
  ),

  -- Latest SAT Marketing Engagement per engagement hash key
  eng_sat_latest AS (
    SELECT * EXCEPT(rn) FROM (
      SELECT
        sme.*,
        ROW_NUMBER() OVER (
          PARTITION BY sme.marketing_engagement_hash_key
          ORDER BY sme.load_date DESC
        ) AS rn
      FROM {catalog_name}.{vault_schema_name}.sat_marketing_engagement sme
    ) WHERE rn = 1
  ),

  -- Engagement hub for engagement_id
  eng_hub AS (
    SELECT
      hme.marketing_engagement_hash_key,
      hme.marketing_engagement_id
    FROM {catalog_name}.{vault_schema_name}.hub_marketing_engagement hme
  ),

  -- -----------------------------
  -- Combine latest Lead + Marketing Preference + Marketing Engagement
  -- -----------------------------
  combined AS (
    SELECT
      hp.person_hash_key,

      -- Marketing Preference
      ph.marketing_preference_id,
      psl.any               AS is_any_communication,
      psl.email_subscriptions AS is_email_subscriptions,
      psl.commercial_email  AS is_commercial_email,
      psl.email             AS is_personal_email,
      psl.call              AS is_call,
      psl.sms               AS is_sms,
      psl.postal_mail       AS is_postal_mail,
      psl.load_date         AS pref_load_date,

      -- Lead (latest)
      ll.preferred_contact_method,
      ll.lead_load_date,

      -- Marketing Engagement
      eh.marketing_engagement_id,
      esl.opened_email      AS is_opened_email,
      esl.marketing_status  AS marketing_status,
      esl.promotion_code    AS promotion_code,
      esl.load_date         AS eng_load_date

    FROM {catalog_name}.{vault_schema_name}.hub_person hp

    LEFT JOIN lead_latest ll
      ON ll.person_hash_key = hp.person_hash_key

    LEFT JOIN pref_link_latest pll
      ON pll.person_hash_key = hp.person_hash_key
    LEFT JOIN pref_hub ph
      ON ph.marketing_preference_hash_key = pll.marketing_preference_hash_key
    LEFT JOIN pref_sat_latest psl
      ON psl.marketing_preference_hash_key = pll.marketing_preference_hash_key

    LEFT JOIN eng_link_latest ell
      ON ell.person_hash_key = hp.person_hash_key
    LEFT JOIN eng_hub eh
      ON eh.marketing_engagement_hash_key = ell.marketing_engagement_hash_key
    LEFT JOIN eng_sat_latest esl
      ON esl.marketing_engagement_hash_key = ell.marketing_engagement_hash_key
  )

  -- -----------------------------
  -- Final projection with effective_from_ts driven by greatest contributing load_date
  -- -----------------------------
  SELECT DISTINCT
    marketing_preference_id,
    is_any_communication,
    preferred_contact_method,
    is_email_subscriptions,
    is_commercial_email,
    is_personal_email,
    is_call,
    is_sms,
    is_postal_mail,

    marketing_engagement_id,
    is_opened_email,
    marketing_status,
    promotion_code,

    to_timestamp(
      greatest(
        COALESCE(lead_load_date, DATE('1900-01-01')),
        COALESCE(pref_load_date, DATE('1900-01-01')),
        COALESCE(eng_load_date,  DATE('1900-01-01'))
      )
    ) AS effective_from_ts,

    CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to_ts

  FROM combined
  WHERE to_timestamp(greatest(
          COALESCE(lead_load_date, DATE('1900-01-01')),
          COALESCE(pref_load_date, DATE('1900-01-01')),
          COALESCE(eng_load_date,  DATE('1900-01-01'))
        )) > (SELECT wm FROM wm)
""",

  "attribute_cols": [
      "marketing_preference_id",
      "is_any_communication",
      "preferred_contact_method",
      "is_email_subscriptions",
      "is_commercial_email",
      "is_personal_email",
      "is_call",
      "is_sms",
      "is_postal_mail",
      "marketing_engagement_id",
      "is_opened_email",
      "marketing_status",
      "promotion_code"
  ],

  "insert_cols": [
      "marketing_preference_id",
      "is_any_communication",
      "preferred_contact_method",
      "is_email_subscriptions",
      "is_commercial_email",
      "is_personal_email",
      "is_call",
      "is_sms",
      "is_postal_mail",
      "marketing_engagement_id",
      "is_opened_email",
      "marketing_status",
      "promotion_code",
      "attr_hash",
      "effective_from_ts",
      "effective_to_ts",
      #"is_current",
      "record_version",
      "created_by",
      "created_ts",
      "last_updated_by",
      "last_updated_ts"
  ]
}